In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import os

CROPPED_ROOT = Path("cropped_dataset")

CLASSES = ['Drought_Trial001_2', 'Watered_Trial001_1']
SPLITS = ['train', 'validation', 'test']

DATA_ROOT = Path("maize_watered_drought_sorted")

yolo_model = YOLO("best.pt")

def extract_and_save_rois():
    if CROPPED_ROOT.exists():
        print("Cropped dataset already exists. Skipping extraction...")
        return

    print("--- Starting YOLO ROI Extraction & Saving ---")
    for split in SPLITS:
        for class_name in CLASSES:
            input_dir = DATA_ROOT / split / class_name
            output_dir = CROPPED_ROOT / split / class_name
            output_dir.mkdir(parents=True, exist_ok=True)
            
            if not input_dir.exists():
                continue
                
            for img_name in os.listdir(input_dir):
                img_path = input_dir / img_name
                img = cv2.imread(str(img_path))
                if img is None:
                    continue
                
                results = yolo_model(img, conf=0.3, iou=0.4, verbose=False)
                
                box_count = 0
                for result in results:
                    for box in result.boxes:
                        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                        
                        # Crop ROI
                        roi = img[y1:y2, x1:x2]
                        
                        if roi.size > 0 and (x2 - x1) > 20 and (y2 - y1) > 20: # Filter tiny false boxes
                            box_count += 1
                            save_path = output_dir / f"{Path(img_name).stem}_crop{box_count}.jpg"
                            cv2.imwrite(str(save_path), roi)
                
                if box_count == 0:
                    fallback_path = output_dir / img_name
                    cv2.imwrite(str(fallback_path), img)

# Execute the extraction step
extract_and_save_rois()

Cropped dataset already exists. Skipping extraction...
